# Task 4.1a. Huấn luyện GCN và GGNN trên Graph A và Graph B

| | |
|---|---|
| **Mục đích** | So sánh biểu diễn AST (Graph A) với AST+CFG+DATA_DEP (Graph B) bằng GCN và GGNN, trên hai protocol |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` (đầu ra của notebook `4_0_build_processed`) |
| **Đầu ra** | `results_4_1a_gnn/`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md` |
| **Máy chạy** | GPU (T4 hoặc P100), bật Internet |

Huấn luyện: class-weighted cross-entropy, chọn epoch tốt nhất theo Macro-F1 trên validation, 3 seed. Kết quả ghi dần nên phiên bị ngắt vẫn giữ được phần đã xong.

In [ ]:
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True, capture_output=True).stdout)

In [ ]:
def run(cmd):
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"Lệnh thất bại với mã {proc.returncode}")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Xác định dữ liệu đầu vào

In [ ]:
found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
         if (p.parent / "project_wise" / "vocab.json").is_file()]
assert found, "Không thấy dữ liệu đã xử lý, hãy Add Input dataset DiverseVul 9077 - Processed Graph Data."
PROCESSED = found[0]
OUT_DIR = Path("/kaggle/working/results_4_1a_gnn")
print("processed:", PROCESSED)

## 2. Chạy thử nhanh

2 epoch, 200 mẫu mỗi split, chỉ để kiểm tra pipeline chạy được. Không dùng để đánh giá.

In [ ]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", "/tmp/g41a/smoke", "--protocols", "project_wise", "--seeds", "42",
     "--limit", "200", "--max-epochs", "2"])

## 3. Huấn luyện đầy đủ

2 graph × 2 model × 2 protocol × 3 seed.

In [ ]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", str(OUT_DIR), "--seeds", "42", "43", "44"])

## 4. Kết quả

Bảng trung bình ± độ lệch chuẩn theo seed. Tệp đầy đủ nằm trong `results_4_1a_gnn/`.

In [ ]:
print((OUT_DIR / "summary.md").read_text(encoding="utf-8"))